# Mask Post-Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliRKhojasteh/Bubble_segmentation/blob/main/Notebooks/mask_postprocessing.ipynb)

Post-processing utilities for bubble segmentation masks:
1. **Overlay visualization** — Blend masks onto original images and save as PNG
2. **MATLAB export** — Convert mask unions to binary `.mat` files for analysis

> Run `automatic_mask_generator.ipynb` first to generate the mask pickle files.

## 1. Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# --- Resolve repository root (works from Notebooks/ or repo root) ---
if IN_COLAB:
    REPO_ROOT = "."
elif os.path.basename(os.getcwd()) == "Notebooks":
    REPO_ROOT = os.path.abspath("..")
else:
    REPO_ROOT = os.path.abspath(".")

# Install scipy if needed (for MATLAB export)
try:
    from scipy.io import savemat
except ImportError:
    %pip install -q scipy
    from scipy.io import savemat

# Download demo data on Colab
DEMO_DIR = os.path.join(REPO_ROOT, "Demo")
if IN_COLAB and not os.path.exists(DEMO_DIR):
    print("Downloading demo data...")
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                   "https://github.com/AliRKhojasteh/Bubble_segmentation.git", "_temp_repo"], 
                  capture_output=True)
    subprocess.run(["git", "-C", "_temp_repo", "sparse-checkout", "set", "Demo"], capture_output=True)
    if os.path.exists("_temp_repo/Demo"):
        import shutil
        shutil.copytree("_temp_repo/Demo", DEMO_DIR)
        shutil.rmtree("_temp_repo")
        print(f"Demo data in '{DEMO_DIR}/'")

print("Setup complete.")

In [ ]:
import pickle
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

## 2. Overlay Visualization

Create PNG images showing the union of all detected bubble masks as a semi-transparent blue overlay on the original experimental image.

In [ ]:
def create_overlaid_png(image_path, pickle_path, output_path=None, alpha=0.3):
    """Create a PNG with all masks overlaid as a blue transparent layer.
    
    Args:
        image_path: Path to the original image (e.g., 'B0001.png').
        pickle_path: Path to the masks pickle file (e.g., 'B0001_masks.pickle').
        output_path: Output PNG path. Defaults to '{base}_overlaid.png'.
        alpha: Overlay transparency (0.0 to 1.0).
    
    Returns:
        Path to the saved overlay PNG.
    """
    image = np.array(Image.open(image_path).convert("RGB"))
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)
    
    with open(pickle_path, "rb") as f:
        masks = pickle.load(f)
    
    if not masks:
        print(f"  No masks for {os.path.basename(image_path)}, saving original.")
        result = image
    else:
        h, w = masks[0]["segmentation"].shape
        binary_mask = np.zeros((h, w), dtype=bool)
        for mask in masks:
            binary_mask |= mask["segmentation"]
        
        overlay = np.zeros((h, w, 4), dtype=np.float32)
        overlay[binary_mask] = [0, 0, 1, alpha]
        
        alpha_ch = overlay[:, :, 3:4]
        blended = (image / 255.0) * (1 - alpha_ch) + overlay[:, :, :3] * alpha_ch
        result = (blended * 255).astype(np.uint8)
    
    if output_path is None:
        base = os.path.splitext(os.path.basename(image_path))[0]
        output_path = os.path.join(os.path.dirname(image_path), f"{base}_overlaid.png")
    
    Image.fromarray(result).save(output_path)
    return output_path

### Batch Generate Overlays

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
IMAGE_DIRECTORY = DEMO_DIR        # Directory with images and pickle files
FILE_PATTERN = "B*.png"           # Image file pattern (excludes _overlaid, _masks files)

# ============================================================

image_paths = sorted(glob.glob(os.path.join(IMAGE_DIRECTORY, FILE_PATTERN)))
# Filter out any already-processed overlay files
image_paths = [p for p in image_paths if "_overlaid" not in p and "_segmented" not in p]

print(f"Processing {len(image_paths)} images...\n")

for image_path in image_paths:
    basename = os.path.splitext(os.path.basename(image_path))[0]
    pickle_path = os.path.join(IMAGE_DIRECTORY, f"{basename}_masks.pickle")
    
    if not os.path.exists(pickle_path):
        print(f"[skip] {basename}: no pickle file")
        continue
    
    out = create_overlaid_png(image_path, pickle_path)
    print(f"[done] {basename} -> {os.path.basename(out)}")

print("\nOverlay generation complete.")

### Preview an Overlay

In [ ]:
# Show the first overlay as a preview
overlaid_files = sorted(glob.glob(os.path.join(IMAGE_DIRECTORY, "*_overlaid.png")))
if overlaid_files:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Find corresponding original
    overlay_path = overlaid_files[0]
    original_name = os.path.basename(overlay_path).replace("_overlaid", "")
    original_path = os.path.join(IMAGE_DIRECTORY, original_name)
    
    if os.path.exists(original_path):
        axes[0].imshow(Image.open(original_path))
        axes[0].set_title("Original")
        axes[0].axis("off")
    
    axes[1].imshow(Image.open(overlay_path))
    axes[1].set_title("With Mask Overlay")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("No overlaid images found. Run the batch generation cell above first.")

## 3. MATLAB Export

Export the union of all bubble masks as binary `.mat` files for analysis in MATLAB (e.g., bubble size distributions, void fraction, equivalent diameter calculations).

The `.mat` file contains a single variable `binary_mask` — a logical 2D array where `true` indicates a bubble pixel.

In [ ]:
def save_binary_masks_matlab(pickle_path, output_path=None):
    """Export the union of all masks as a binary MATLAB .mat file.
    
    Args:
        pickle_path: Path to the masks pickle file.
        output_path: Output .mat path. Defaults to '{dir}/matlab/{base}_binary.mat'.
    
    Returns:
        Path to the saved .mat file.
    """
    with open(pickle_path, "rb") as f:
        masks = pickle.load(f)
    
    if not masks:
        print(f"  No masks in {os.path.basename(pickle_path)}.")
        binary_mask = np.array([], dtype=bool)
    else:
        h, w = masks[0]["segmentation"].shape
        binary_mask = np.zeros((h, w), dtype=bool)
        for mask in masks:
            binary_mask |= mask["segmentation"]
    
    if output_path is None:
        base = os.path.splitext(os.path.basename(pickle_path))[0].replace("_masks", "")
        matlab_dir = os.path.join(os.path.dirname(pickle_path), "matlab")
        os.makedirs(matlab_dir, exist_ok=True)
        output_path = os.path.join(matlab_dir, f"{base}_binary.mat")
    
    savemat(output_path, {"binary_mask": binary_mask})
    return output_path

### Batch MATLAB Export

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
IMAGE_DIRECTORY = DEMO_DIR

# ============================================================

pickle_files = sorted(glob.glob(os.path.join(IMAGE_DIRECTORY, "*_masks.pickle")))
print(f"Exporting {len(pickle_files)} mask files to MATLAB...\n")

for pickle_path in pickle_files:
    out = save_binary_masks_matlab(pickle_path)
    basename = os.path.basename(pickle_path).replace("_masks.pickle", "")
    print(f"[done] {basename} -> {os.path.basename(out)}")

print(f"\nMATLAB export complete. Files saved in '{IMAGE_DIRECTORY}/matlab/'")

## MATLAB Usage

To load the binary masks in MATLAB:
```matlab
data = load('B0001_binary.mat');
binary_mask = data.binary_mask;

% Display
imshow(binary_mask);
title('Bubble Binary Mask');

% Compute properties
props = regionprops(binary_mask, 'Area', 'EquivDiameter', 'Centroid');
diameters = [props.EquivDiameter];
fprintf('Detected %d bubbles, mean diameter: %.1f px\n', length(props), mean(diameters));
```